# Data Preprocessing

In [57]:
import pandas as pd
import numpy as np
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head(3)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN


In [58]:
df = df.drop(columns=['id', 'Unnamed: 32'])

In [59]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [60]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [61]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [62]:
# Convert numpy arrays to PyTorch tensors
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.float32)
y_test = torch.from_numpy(y_test).to(torch.float32)

# Defining the model

In [66]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 1),  #Linear(#_inputs, #_outputs)
        nn.Sigmoid()
    )

  def forward(self, num_features):
    out = self.network(num_features)
    return out

In [67]:
learning_rate = 0.1
epochs = 25

In [68]:
model = Neural_Network(X_train.shape[1])
loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
  y_pred = model(X_train)
  loss = loss_function(y_pred, y_train.reshape(-1, 1))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:1.1127808094024658
Epoch: 2, Loss:0.7881291508674622
Epoch: 3, Loss:0.6083311438560486
Epoch: 4, Loss:0.5066459774971008
Epoch: 5, Loss:0.4413076639175415
Epoch: 6, Loss:0.3951934576034546
Epoch: 7, Loss:0.36055511236190796
Epoch: 8, Loss:0.3333798050880432
Epoch: 9, Loss:0.3113689720630646
Epoch: 10, Loss:0.2931029796600342
Epoch: 11, Loss:0.2776523530483246
Epoch: 12, Loss:0.26438015699386597
Epoch: 13, Loss:0.25283369421958923
Epoch: 14, Loss:0.24268127977848053
Epoch: 15, Loss:0.23367370665073395
Epoch: 16, Loss:0.22561950981616974
Epoch: 17, Loss:0.21836881339550018
Epoch: 18, Loss:0.21180245280265808
Epoch: 19, Loss:0.2058241218328476
Epoch: 20, Loss:0.2003551870584488
Epoch: 21, Loss:0.19533057510852814
Epoch: 22, Loss:0.19069595634937286
Epoch: 23, Loss:0.18640559911727905
Epoch: 24, Loss:0.1824207454919815
Epoch: 25, Loss:0.1787082701921463


In [74]:
model.network

Sequential(
  (0): Linear(in_features=30, out_features=1, bias=True)
  (1): Sigmoid()
)

In [72]:
print(model.network[0].weight)

Parameter containing:
tensor([[ 0.2643,  0.1850,  0.1759,  0.2335, -0.0617,  0.2881,  0.1547,  0.3293,
          0.0674, -0.1868,  0.2519, -0.1477,  0.3179,  0.0521,  0.0512,  0.0973,
          0.0711,  0.0071,  0.1020, -0.1793,  0.3127,  0.1774,  0.1841,  0.1836,
          0.2765,  0.1384,  0.1716,  0.1928,  0.0187,  0.1574]],
       requires_grad=True)


In [73]:
print(model.network[0].bias)

Parameter containing:
tensor([-0.0774], requires_grad=True)

In [77]:
# Evaluation
with torch.no_grad():
  y_pred = model.forward(X_test)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy: {accuracy}')

Accuracy: 0.5861803889274597
